# 🎙️ Audio-Automation: Chatterbox V3 Multilingual GPU Production
**Genre:** 💔 Desi Love, Romance & Emotional Twist Drama
**Engine:** Chatterbox Multilingual V3 + Step 11 Cinema Quality DSP (No Edge TTS)

In [ ]:
# 1. GPU & Environment Setup (Proven Stack)
import torch, os
assert torch.cuda.is_available(), 'GPU enable karo'
print('GPU:', torch.cuda.get_device_name(0))

!sudo apt-get update -qq || true
!sudo apt-get install -y -qq ffmpeg sox libsox-dev build-essential python3-dev > /dev/null

!pip install -q --upgrade pip wheel setuptools
!echo 'numpy<2.0.0' > /tmp/constraint.txt
os.environ['PIP_CONSTRAINT'] = '/tmp/constraint.txt'
!pip install -q --no-cache-dir numpy==1.26.4

# Chatterbox TTS & Dependencies
!PIP_CONSTRAINT='' pip install -q --no-cache-dir conformer==0.3.2 pykakasi==2.3.0 pyloudnorm s3tokenizer diffusers==0.29.0 safetensors==0.5.3
!pip install -q --no-cache-dir resemble-perth sox vocos einops spacy-pkuseg
!pip install -q --no-cache-dir --no-deps git+https://github.com/resemble-ai/chatterbox.git
!pip install -q --no-cache-dir pydub soundfile scipy librosa
!pip install -q --no-cache-dir --force-reinstall transformers==5.2.0 tokenizers==0.21.1
!pip install -q --no-cache-dir --force-reinstall numpy==1.26.4

print('✅ Environment Setup Complete!')

In [ ]:
# 2. Load Chatterbox Multilingual V3 Model
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading Chatterbox Multilingual V3 on {device}...')
model = ChatterboxMultilingualTTS.from_pretrained(device=device, t3_model='v3')
print('✅ Chatterbox Multilingual V3 successfully loaded!')

In [ ]:
# 3. Audio Enhancement (DSP from Step 11)
import numpy as np, soundfile as sf
from scipy import signal

def enhance_voice_dsp(audio, sr=24000):
    # Presence EQ (3kHz boost)
    w0 = 2 * np.pi * 3000.0 / sr
    alpha = np.sin(w0) / (2.0 * (2000.0 / sr))
    A = 10.0 ** (4.5 / 40.0)
    b = np.array([1.0 + alpha * A, -2.0 * np.cos(w0), 1.0 - alpha * A])
    a = np.array([1.0 + alpha / A, -2.0 * np.cos(w0), 1.0 - alpha / A])
    audio = signal.filtfilt(b / a[0], a / a[0], audio)
    
    # Loudness normalize (-23 LUFS)
    rms = np.sqrt(np.mean(audio**2))
    gain = 10.0 ** ((-23.0 - (20 * np.log10(max(rms, 1e-10)))) / 20.0)
    audio = audio * gain
    
    # Peak limit
    peak = np.max(np.abs(audio))
    if peak > 0.95:
        audio = audio * (0.94 / peak)
    return audio.astype(np.float32)
print('✅ Cinema Quality DSP loaded!')

In [ ]:
# 4. Generate Speech & Master Audio
sample_text = 'आरुषि... काश उस रात मैंने तुम्हारी बात सुन ली होती। आज तुम किसी और की हो, और मैं सिर्फ तुम्हारी यादों का क़ैदी।'
print('Generating dialogue with emotional nuance...')
wav = model.generate(sample_text, language_id='hi', exaggeration=0.75, cfg_weight=0.20)
if hasattr(wav, 'cpu'):
    wav = wav.cpu().numpy().squeeze()

enhanced = enhance_voice_dsp(wav, sr=model.sr)
sf.write('master_output.wav', enhanced, model.sr, subtype='PCM_16')

import IPython.display as ipd
ipd.Audio('master_output.wav')